# Week 9 · Module Demo — Transformer from Scratch
### NaijaLingo: Building a Mini Translation Engine, One Mechanism at a Time

**The story:** you're building the first prototype for **NaijaLingo**, a Nigerian language-tech
startup. Before your team reaches for an off-the-shelf Transformer library, your lead asked for
one thing first: *prove you understand what's actually happening inside one.* This notebook is
that proof — every mechanism a real translation model relies on, built from scratch, in order,
on a real sentence pair.

**Running example throughout:** English `"I am going to the market"` → Yoruba
`"Mo n lọ si ọja"`.

This notebook doesn't introduce anything new — every function here was built and explained,
step by step, in Topics 1, 2, and 4's demo notebooks. What's new is seeing them all run
together, in the order a real model would actually use them:

1. **Tokenize** the sentence pair
2. **Scaled dot-product attention**, computed by hand (Topic 1)
3. **Multi-head attention + positional encoding** (Topic 2)
4. **A full, stacked Transformer block** — feed-forward, residual connections, layer
   normalisation (Topic 4)
5. **Inspect the final output** — what NaijaLingo's encoder would actually hand off to a
   decoder next

See `README.md` for exactly how each section below maps back to its topic.

In [ ]:
import numpy as np
import math

np.set_printoptions(precision=3, suppress=True)
print("NumPy and math ready. Let's build NaijaLingo's encoder.")


NumPy and math ready. Let's build NaijaLingo's encoder.


## Section 1 — Tokenize the Sentence Pair

Before any math happens, NaijaLingo's pipeline needs the sentence broken into tokens, with each
token given a starting numeric representation (an embedding).

In [ ]:
tokens_full = ["I", "am", "going", "to", "the", "market"]
yoruba_reference = "Mo n lọ si ọja"

d_model = 8   # size of each token's embedding

np.random.seed(7)
embeddings_full = {t: np.round(np.random.randn(d_model), 2) for t in tokens_full}
X_full = np.array([embeddings_full[t] for t in tokens_full])

print("English tokens: ", tokens_full)
print("Yoruba reference:", yoruba_reference)
print()
print("Embeddings:")
for t in tokens_full:
    print(f"  {t:8s} -> {embeddings_full[t]}")
print()
print("Embedding matrix shape:", X_full.shape, "(6 tokens x 8 dimensions)")


English tokens:  ['I', 'am', 'going', 'to', 'the', 'market']
Yoruba reference: Mo n lọ si ọja

Embeddings:
  I        -> [ 1.69 -0.47  0.03  0.41 -0.79  0.   -0.   -1.75]
  am       -> [ 1.02  0.6  -0.63 -0.17  0.51 -0.26 -0.24 -1.45]
  going    -> [ 0.55  0.12  0.27 -1.53  1.65  0.15 -0.39  2.03]
  to       -> [-0.05 -1.45 -0.41 -2.29  1.05 -0.42 -0.74  1.07]
  the      -> [-1.65  0.54 -2.06 -0.66 -1.2   1.46  1.77 -0.33]
  market   -> [ 0.84 -0.18  0.57 -0.75 -1.71 -1.8   0.38  2.25]

Embedding matrix shape: (6, 8) (6 tokens x 8 dimensions)


**Important distinction, carried over from Topic 1:** this notebook computes
**self-attention** within the English sentence. The Yoruba sentence is kept as reference context
throughout — it's *why* NaijaLingo needs this pipeline, not something the code computes
attention against directly. (Attention *between* the two languages — cross-attention — is the
mechanism a full encoder-decoder model would use, covered conceptually in Topic 3.)

## Section 2 — Scaled Dot-Product Attention, By Hand (Topic 1)

Every function in this section is reused, unchanged, from Topic 1's notebook.

In [ ]:
# --- Reused from Topic 1 ---
def softmax(scores):
    max_score = max(scores)
    exp_scores = [math.exp(s - max_score) for s in scores]
    total = sum(exp_scores)
    return [e / total for e in exp_scores]

def softmax_rows(matrix):
    return np.array([softmax(list(row)) for row in matrix])

def scaled_dot_product_attention(X, W_Q, W_K, W_V):
    """Score -> Scale -> Normalise -> Blend, exactly as built in Topic 1."""
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k = Q.shape[-1]
    raw_scores = Q @ K.T
    scaled_scores = raw_scores / math.sqrt(d_k)
    weights = softmax_rows(scaled_scores)
    output = weights @ V
    return output, weights

print("Topic 1's attention function is ready.")


Topic 1's attention function is ready.


In [ ]:
# Run single-head self-attention on NaijaLingo's sentence.
np.random.seed(11)
W_Q = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_K = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_V = np.round(np.random.randn(d_model, d_model) * 0.3, 2)

single_head_output, single_head_weights = scaled_dot_product_attention(X_full, W_Q, W_K, W_V)

print("Single-head attention weights (rows = Query word, columns = Key word):")
header = "         " + "".join(f"{t:>8s}" for t in tokens_full)
print(header)
for tok, row in zip(tokens_full, single_head_weights):
    print(f"{tok:8s} " + "".join(f"{v:8.3f}" for v in row))


Single-head attention weights (rows = Query word, columns = Key word):
                I      am   going      to     the  market
I           0.236   0.204   0.019   0.044   0.466   0.031
am          0.190   0.243   0.086   0.143   0.262   0.076
going       0.112   0.109   0.182   0.378   0.006   0.213
to          0.066   0.042   0.141   0.202   0.018   0.531
the         0.024   0.029   0.493   0.047   0.296   0.112
market      0.192   0.159   0.140   0.264   0.034   0.212


**Checkpoint:** this is exactly Topic 1's mechanism — one attention head, comparing every
word in `"I am going to the market"` against every other word. It's a working building block,
but as Topic 2 pointed out, one head can only capture one kind of relationship, and this
mechanism has no idea what order the words came in. NaijaLingo's pipeline needs more.

## Section 3 — Multi-Head Attention + Positional Encoding (Topic 2)

Every function in this section is reused, unchanged, from Topic 2's notebook.

In [ ]:
# --- Reused from Topic 2 ---
def split_into_heads(matrix, num_heads):
    head_dim = matrix.shape[1] // num_heads
    heads = []
    for h in range(num_heads):
        start = h * head_dim
        end = start + head_dim
        one_head = np.array([row[start:end] for row in matrix])
        heads.append(one_head)
    return heads

def concatenate_heads(head_outputs):
    num_tokens = head_outputs[0].shape[0]
    combined = []
    for i in range(num_tokens):
        row = []
        for head_output in head_outputs:
            row.extend(list(head_output[i]))
        combined.append(row)
    return np.array(combined)

def attention_from_qkv(Q, K, V):
    d_k = Q.shape[-1]
    raw_scores = Q @ K.T
    scaled_scores = raw_scores / math.sqrt(d_k)
    weights = softmax_rows(scaled_scores)
    output = weights @ V
    return output, weights

def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    Q_heads = split_into_heads(Q, num_heads)
    K_heads = split_into_heads(K, num_heads)
    V_heads = split_into_heads(V, num_heads)
    outputs, all_weights = [], []
    for h in range(num_heads):
        out_h, w_h = attention_from_qkv(Q_heads[h], K_heads[h], V_heads[h])
        outputs.append(out_h)
        all_weights.append(w_h)
    combined = concatenate_heads(outputs)
    output = combined @ W_O
    return output, all_weights

def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for dim in range(d_model):
            angle = pos / (10000 ** (2 * (dim // 2) / d_model))
            if dim % 2 == 0:
                pe[pos][dim] = math.sin(angle)
            else:
                pe[pos][dim] = math.cos(angle)
    return pe

print("Topic 2's multi-head attention and positional encoding functions are ready.")


Topic 2's multi-head attention and positional encoding functions are ready.


In [ ]:
# Add positional encoding, then run multi-head attention.
PE = positional_encoding(len(tokens_full), d_model)
X_pos = X_full + PE   # position-aware input

num_heads = 2
W_O = np.round(np.random.randn(d_model, d_model) * 0.3, 2)

mha_output, mha_weights = multi_head_attention(X_pos, W_Q, W_K, W_V, W_O, num_heads)

print("Multi-head attention output, with positional encoding:")
for tok, vec in zip(tokens_full, mha_output):
    print(f"  {tok:8s} -> {vec}")
print()
print("Output shape:", mha_output.shape)


Multi-head attention output, with positional encoding:
  I        -> [-0.511 -0.386 -0.03  -0.359  0.009  0.352  0.346 -0.08 ]
  am       -> [-0.488 -0.008 -0.447 -0.274  0.076  0.432  0.458 -0.372]
  going    -> [-0.403  0.323 -1.005 -0.014  0.26   0.326  0.211 -0.6  ]
  to       -> [-0.396  0.081 -0.504  0.12  -0.007  0.415  0.391 -0.265]
  the      -> [-0.725 -1.041  0.993 -0.215 -0.77   0.864  1.061  0.662]
  market   -> [-0.456  0.16  -0.729 -0.032  0.089  0.453  0.396 -0.445]

Output shape: (6, 8)


**Checkpoint:** NaijaLingo's encoder now has two heads catching different relationships in
the sentence, and every word's representation reflects *where* it sits, not just what it means.
This is a meaningfully richer representation than Section 2's single-head output — but it's
still only one "layer" of processing. A real encoder needs a feed-forward network and the
ability to stack multiple layers deep, which is where Topic 4 comes in.

## Section 4 — A Full, Stacked Transformer Block (Topic 4)

Every function in this section is reused, unchanged, from Topic 4's notebook.

In [ ]:
# --- Reused from Topic 4 ---
def layer_norm(X, eps=1e-6):
    normalized_rows = []
    for row in X:
        mean = sum(row) / len(row)
        variance = sum((value - mean) ** 2 for value in row) / len(row)
        std = math.sqrt(variance + eps)
        normalized_rows.append([(value - mean) / std for value in row])
    return np.array(normalized_rows)

def feed_forward(X, W1, b1, W2, b2):
    hidden = X @ W1 + b1
    hidden_relu = np.maximum(0, hidden)
    return hidden_relu @ W2 + b2

def transformer_block(X, W_Q, W_K, W_V, W_O, num_heads, W1, b1, W2, b2):
    attn_output, attn_weights = multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads)
    X_after_attention = layer_norm(X + attn_output)
    ff_output = feed_forward(X_after_attention, W1, b1, W2, b2)
    X_after_ff = layer_norm(X_after_attention + ff_output)
    return X_after_ff, attn_weights

print("Topic 4's layer_norm(), feed_forward(), and transformer_block() are ready.")


Topic 4's layer_norm(), feed_forward(), and transformer_block() are ready.


In [ ]:
# Generate independent weights for 2 stacked blocks -- NaijaLingo's encoder,
# 2 layers deep.
def random_block_weights(d_model, d_ff, seed):
    rng = np.random.RandomState(seed)
    W_Q = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W_K = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W_V = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W_O = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W1 = np.round(rng.randn(d_model, d_ff) * 0.3, 2)
    b1 = np.zeros(d_ff)
    W2 = np.round(rng.randn(d_ff, d_model) * 0.3, 2)
    b2 = np.zeros(d_model)
    return W_Q, W_K, W_V, W_O, W1, b1, W2, b2

d_ff = 16
num_blocks = 2
block_params = [random_block_weights(d_model, d_ff, seed=100 + i) for i in range(num_blocks)]
print(f"Generated independent weights for {num_blocks} stacked blocks.")


Generated independent weights for 2 stacked blocks.


In [ ]:
# Feed the position-aware input through NaijaLingo's 2-block encoder stack.
X_stacked = X_pos
for i, params in enumerate(block_params):
    W_Q_i, W_K_i, W_V_i, W_O_i, W1_i, b1_i, W2_i, b2_i = params
    X_stacked, _ = transformer_block(
        X_stacked, W_Q_i, W_K_i, W_V_i, W_O_i, num_heads, W1_i, b1_i, W2_i, b2_i
    )
    print(f"After block {i + 1}: shape {X_stacked.shape}")


After block 1: shape (6, 8)
After block 2: shape (6, 8)


## Section 5 — Inspect the Final Output

This is what NaijaLingo's encoder would hand off next — to a decoder's cross-attention sublayer
(Topic 3), which would use it to generate the Yoruba translation one word at a time.

In [ ]:
print("Final encoder output for:", " ".join(tokens_full))
print("(Yoruba reference:", yoruba_reference, ")\n")

for tok, vec in zip(tokens_full, X_stacked):
    print(f"  {tok:8s} -> {vec}")

print("\nFinal shape:", X_stacked.shape, "-- still 6 tokens x 8 dimensions,")
print("but now each vector carries: word meaning + word order + gathered context")
print("from 2 full layers of attention and feed-forward processing.")


Final encoder output for: I am going to the market
(Yoruba reference: Mo n lọ si ọja )

  I        -> [ 1.88  -1.442  0.954 -0.777 -0.511  0.6   -0.392 -0.313]
  am       -> [ 0.138  1.274 -1.465  0.057  1.208 -1.307 -0.671  0.768]
  going    -> [ 0.006 -0.238 -0.4   -1.267  0.559 -0.846 -0.082  2.268]
  to       -> [-0.013 -1.446  0.602 -0.933  0.846 -0.711 -0.194  1.849]
  the      -> [ 0.407 -0.377  1.483 -0.911 -1.303  1.278  0.475 -1.052]
  market   -> [ 1.056 -0.768  0.674 -0.81  -1.282 -0.857  0.349  1.638]

Final shape: (6, 8) -- still 6 tokens x 8 dimensions,
but now each vector carries: word meaning + word order + gathered context
from 2 full layers of attention and feed-forward processing.


In [ ]:
# One last sanity check: compare the very first representation (raw embeddings)
# to the final one (after tokenizing, positional encoding, and 2 stacked blocks),
# to see how much each word's representation actually moved.
print("How much did each word's representation change end-to-end?")
for tok, original, final in zip(tokens_full, X_full, X_stacked):
    difference = np.abs(final - original).mean()
    print(f"  {tok:8s} -> mean absolute change: {difference:.3f}")


How much did each word's representation change end-to-end?
  I        -> mean absolute change: 0.748
  am       -> mean absolute change: 0.877
  going    -> mean absolute change: 0.559
  to       -> mean absolute change: 0.529
  the      -> mean absolute change: 1.134
  market   -> mean absolute change: 0.373


**What this confirms:** every word's representation changed substantially from where it
started — not randomly, but through the exact sequence of well-defined operations this week
built: attention gathering context, positional encoding injecting order, and two full
Transformer blocks refining the result further. This is the representation a real encoder would
produce — and exactly what a decoder would need to generate NaijaLingo's Yoruba translation.

## Wrap-Up

Starting from `"I am going to the market"`, this notebook:

1. Tokenized the sentence and gave each word a starting embedding (Section 1)
2. Computed single-head scaled dot-product attention by hand, reusing Topic 1's exact function
   (Section 2)
3. Extended to multi-head attention and added positional encoding, reusing Topic 2's exact
   functions (Section 3)
4. Assembled and stacked 2 complete Transformer blocks — attention, residual, norm,
   feed-forward, residual, norm — reusing Topic 4's exact functions (Section 4)
5. Inspected the final output vectors NaijaLingo's encoder would hand off to a decoder next
   (Section 5)

Every function used here was built from scratch, understood one piece at a time, across
Topics 1, 2, and 4 — nothing in this notebook was new math, only a new order to run it in.

See `README.md` for the full section-by-section map back to each topic's slides and demo
notebook.